In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:14:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Reading in the external ABS data as business indicators  
from urllib.request import urlretrieve
import os

# SEIFA 2021, SA2 level -- gives IRSAD score/decile per SA2 (socio-economic index)
# source: https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021
SEIFA_FILES = {
    "SEIFA_2021_SA2.xlsx": "https://www.abs.gov.au/statistics/people/people-and-communities/socio-economic-indexes-areas-seifa-australia/2021/Statistical%20Area%20Level%202%2C%20Indexes%2C%20SEIFA%202021.xlsx",
}


output_relative_dir = '../../data/'

# data output directory is
abs_output_dir = output_relative_dir + 'raw_abs'
if not os.path.exists(abs_output_dir):
    os.makedirs(abs_output_dir)


# def download_files(file_dict, output_dir):
#     for filename, url in file_dict.items():
#         output_path = f"{output_dir}/{filename}"

#          # Checks if the files have already been downloaded, skip if we've already downloaded it, 
#         # every time the notebook is re-run for reproducibility checks
#         if os.path.exists(output_path):
#             print(f"Already downloaded: {filename}")
#             continue

#         print(f"Downloading {filename}...")
#         urlretrieve(url, output_path)
#         print(f"Completed {filename}")

# download_files(SEIFA_FILES, abs_output_dir)

for filename, url in SEIFA_FILES.items():
    output_path = f"{abs_output_dir}/{filename}"

    # Checks if the files have already been downloaded, skip if we've already downloaded it, 
    # every time the notebook is re-run for reproducibility checks
    if os.path.exists(output_path):
        print(f"Already downloaded: {filename}")
        continue

    print(f"Downloading {filename}...")
    urlretrieve(url, output_path)
    print(f"Completed {filename}")

Already downloaded: SEIFA_2021_SA2.xlsx


In [3]:
import pandas as pd
excel_path_seifa = f"{abs_output_dir}/SEIFA_2021_SA2.xlsx"
print(pd.ExcelFile(excel_path_seifa).sheet_names)


['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Explanatory Notes']


In [4]:
# Check the "Contents" sheet
pd.read_excel(excel_path_seifa, sheet_name='Contents', header=None)

,0,1,2
0,Australian Bureau of Statistics,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN
3,NaN,NaN,NaN
4,NaN,Contents,NaN
5,NaN,Tables,NaN
6,NaN,1,"Statistical Area Level 2 (SA2) SEIFA Summary, ..."
7,NaN,2,Statistical Area Level 2 (SA2) Index of Relati...
8,NaN,3,Statistical Area Level 2 (SA2) Index of Relati...
9,NaN,4,Statistical Area Level 2 (SA2) Index of Econom...


In [5]:
# Check the "Table 1" sheet
pd.read_excel(excel_path_seifa, sheet_name='Table 1', header=None, nrows=15)

,0,1,2,3,4,5,6,7,8,9,10
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 1 Statistical Area Level 2 (SA2) SEIFA S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Index of Relative Socio-economic Disadvantage,NaN,Index of Relative Socio-economic Advantage and...,NaN,Index of Economic Resources,NaN,Index of Education and Occupation,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Score,Decile,Score,Decile,Score,Decile,Score,Decile,Usual Resident Population
6,101021007,Braidwood,1024,6,1001,6,1027,7,1008,6,4343
7,101021008,Karabar,994,5,982,5,1000,5,967,5,8517
8,101021009,Queanbeyan,1010,5,998,6,945,3,1000,6,11342
9,101021010,Queanbeyan - East,1025,6,1015,6,969,4,1025,7,5085


In [6]:
# Drop the top 5 rows that are unnecessary for the analysis 
seifa_table1 = pd.read_excel(excel_path_seifa, sheet_name='Table 1', skiprows=5)


seifa_table1.columns = [
    'SA2_CODE_2021',
    'SA2_NAME_2021',
    'IRSD_score', 'IRSD_decile',
    'IRSAD_score', 'IRSAD_decile',
    'IER_score', 'IER_decile',
    'IEO_score', 'IEO_decile',
    'usual_resident_population',
]

seifa_irsad = seifa_table1[['SA2_CODE_2021', 'SA2_NAME_2021', 'IRSAD_score', 'IRSAD_decile']]

In [ ]:
# Check the bottom of the df for any trailing details that are not part of the data 
seifa_irsad.tail(10)

# Drop trailing bits 
seifa_irsad = seifa_irsad.iloc[:-2] # Drop last two rows since they are not part of the data 
seifa_irsad.tail(10

,SA2_CODE_2021,SA2_NAME_2021,IRSAD_score,IRSAD_decile
2356,801091110,Torrens,1112,10
2357,801101135,Coombs,1117,10
2358,801101136,Denman Prospect,1174,10
2359,801101139,Wright,1142,10
2360,801111140,ACT - South West,1110,9
2361,801111141,Namadgi,932,3
2362,901011001,Christmas Island,972,4
2363,901021002,Cocos (Keeling) Islands,903,2
2364,901031003,Jervis Bay,905,2
2365,901041004,Norfolk Island,958,4


In [8]:
seifa_excluded = pd.read_excel(excel_path_seifa, sheet_name='Table 6', header=None, nrows=15)
seifa_excluded 

# seifa_excluded 

,0,1,2,3,4,5,6
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN
1,"Socio-Economic Indexes for Australia (SEIFA), ...",NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 27 April 2023,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 6 Statistical Area Level 2 (SA2) Exclude...,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,Area did not receive an index score,NaN,NaN,NaN
5,2021 Statistical Area Level 2 (SA2) 9-Digit Code,2021 Statistical Area Level 2 (SA2) Name,Usual Resident Population,IRSD,IRSAD,IER,IEO
6,103031075,Wollangambe - Wollemi,0,Y,Y,Y,Y
7,107011133,Port Kembla Industrial,5,Y,Y,Y,Y
8,107021135,Illawarra Catchment Reserve,9,Y,Y,Y,Y
9,111031230,Newcastle Port - Kooragang,31,Y,Y,N,N


In [10]:
seifa_irsad.head()
seifa_irsad.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2366 entries, 0 to 2365
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   SA2_CODE_2021  2366 non-null   object
 1   SA2_NAME_2021  2366 non-null   object
 2   IRSAD_score    2366 non-null   object
 3   IRSAD_decile   2366 non-null   object
dtypes: object(4)
memory usage: 74.1+ KB


In [11]:
seifa_irsad['IRSAD_score'].apply(type).value_counts()

IRSAD_score
<class 'int'>    2353
<class 'str'>      13
Name: count, dtype: int64

In [ ]:
seifa_irsad['IRSAD_score'].isna().sum() # Counting the number of rows with missing IRSAD_score

np.int64(13)

In [12]:
mask = seifa_irsad['IRSAD_score'].apply(lambda x: isinstance(x, str))
seifa_irsad.loc[mask]

,SA2_CODE_2021,SA2_NAME_2021,IRSAD_score,IRSAD_decile
234,111031230,Newcastle Port - Kooragang,-,-
602,127021521,Wetherill Park Industrial,-,-
624,128021537,Royal National Park,-,-
724,205031092,Wilsons Promontory,-,-
732,205051099,Alps - West,-,-
831,208031184,Braeside,-,-
1758,403041082,Lonsdale,-,-
1978,506031130,Welshpool,-,-
1997,507011150,Bibra Industrial,-,-
2014,507031173,Kwinana Industrial,-,-


In [14]:
seifa_irsad['IRSAD_score'] = pd.to_numeric(seifa_irsad['IRSAD_score'], errors='coerce')
seifa_irsad['IRSAD_decile'] = pd.to_numeric(seifa_irsad['IRSAD_decile'], errors='coerce')

In [15]:
seifa_irsad['IRSAD_score'].apply(type).value_counts()

IRSAD_score
<class 'float'>    2366
Name: count, dtype: int64

In [25]:
# 2021 Census DataPack 
# Source: https://www.abs.gov.au/census/find-census-data/datapacks
# Data was donwlaoded manually as a zip file from source 
income_zip_path = f"{abs_output_dir}/2021_GCP_all_for_AUS"

assert os.path.exists(income_zip_path), (
    "Census DataPack zip not found, manually download data from source, "
    f"save it to {income_zip_path}"
)

In [17]:
# Set the manual directory pathway that will lead to the SA2 dataset for Australia within the census data pack 
sa2_dir = f"{abs_output_dir}/2021_GCP_all_for_AUS/2021 Census GCP All Geographies for AUS/SA2/AUS"

# From the SA2 data inside the data pack for AUS, we only need the data pertaining to GO2
# GO2 contains the data on median household income, median weekly income, etc. 
g02_sa2_data = [
    os.path.join(sa2_dir, f)
    for f in os.listdir(sa2_dir)
    if 'G02' in f.upper() and f.lower().endswith('.csv')
]
print(g02_sa2_data) # Produces a list 

['../../data/raw_abs/2021_GCP_all_for_AUS/2021 Census GCP All Geographies for AUS/SA2/AUS/2021Census_G02_AUST_SA2.csv']


In [18]:
income_path = g02_sa2_data[0] # Saves the list element as a string 
print(income_path)

g02_sa2_raw = pd.read_csv(income_path) # Save the csv which is led to by the string 
g02_sa2_raw.head()

../../data/raw_abs/2021_GCP_all_for_AUS/2021 Census GCP All Geographies for AUS/SA2/AUS/2021Census_G02_AUST_SA2.csv


,SA2_CODE_2021,Median_age_persons,Median_mortgage_repay_monthly,Median_tot_prsnl_inc_weekly,Median_rent_weekly,Median_tot_fam_inc_weekly,Average_num_psns_per_bedroom,Median_tot_hhd_inc_weekly,Average_household_size
0,101021007,51,1732,760,330,1886,0.8,1429,2.2
1,101021008,38,1950,975,350,2334,0.8,1989,2.6
2,101021009,37,1700,996,330,2233,0.9,1703,2.1
3,101021010,36,1700,1104,310,2412,0.9,1796,2.1
4,101021012,37,2300,1357,430,3332,0.8,3014,2.9


In [19]:
# Merge the seifa_irsad and go2_sa2_raw on common column SA2_CODE_2021

# Specify the data type of SA2_CODE_2021 as a string instead of an integer for future uses 
seifa_irsad['SA2_CODE_2021'] = seifa_irsad['SA2_CODE_2021'].astype(str)
g02_sa2_raw['SA2_CODE_2021'] = g02_sa2_raw['SA2_CODE_2021'].astype(str)


features_sa2_df = seifa_irsad.merge(g02_sa2_raw, on='SA2_CODE_2021', how='outer', indicator=True)
print(features_sa2_df['_merge'].value_counts())  # sanity check: how many SA2s matched cleanly

features_sa2_df = features_sa2_df.drop(columns='_merge') # _merge specifies how the join behaves, not necessary 
features_sa2_df.head()

_merge
both          2366
right_only     106
left_only        0
Name: count, dtype: int64


,SA2_CODE_2021,SA2_NAME_2021,IRSAD_score,IRSAD_decile,Median_age_persons,Median_mortgage_repay_monthly,Median_tot_prsnl_inc_weekly,Median_rent_weekly,Median_tot_fam_inc_weekly,Average_num_psns_per_bedroom,Median_tot_hhd_inc_weekly,Average_household_size
0,101021007,Braidwood,1001.0,6.0,51,1732,760,330,1886,0.8,1429,2.2
1,101021008,Karabar,982.0,5.0,38,1950,975,350,2334,0.8,1989,2.6
2,101021009,Queanbeyan,998.0,6.0,37,1700,996,330,2233,0.9,1703,2.1
3,101021010,Queanbeyan - East,1015.0,6.0,36,1700,1104,310,2412,0.9,1796,2.1
4,101021012,Queanbeyan West - Jerrabomberra,1107.0,9.0,37,2300,1357,430,3332,0.8,3014,2.9


In the above join we can identify total of 2366 rows that are present in both datasets and merged cleanly with our SA2_CODE_2021 as the key. However, there are an additional 106 rows that exists in the g02_sa2_raw dataset (income dataset) but they do have a matching seifa score (right_only=0). This is probably due to decisions made in the data collection as SEIFA is calculated because it is a calculated index requiring a minimum population to produce a reliable score. ABS excludes areas with too few or no residents (mostly airports, parks, and industrial zones, this is seen in SEIFA workbook's Table 6). 

The income table has no such requirement and reports for all 2,472 standard SA2s regardless of population.
Census income table (which g02_sa2_raw comes from) still counts every SA2, therefore, no SA2 exists in seifa_irsad without a matching income value, so the income dataset fully covers SEIFA's geography (left_only=0)

In [ ]:
print(features_sa2_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2472 entries, 0 to 2471
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   SA2_CODE_2021                  2472 non-null   object 
 1   SA2_NAME_2021                  2366 non-null   object 
 2   IRSAD_score                    2353 non-null   float64
 3   IRSAD_decile                   2353 non-null   float64
 4   Median_age_persons             2472 non-null   int64  
 5   Median_mortgage_repay_monthly  2472 non-null   int64  
 6   Median_tot_prsnl_inc_weekly    2472 non-null   int64  
 7   Median_rent_weekly             2472 non-null   int64  
 8   Median_tot_fam_inc_weekly      2472 non-null   int64  
 9   Average_num_psns_per_bedroom   2472 non-null   float64
 10  Median_tot_hhd_inc_weekly      2472 non-null   int64  
 11  Average_household_size         2472 non-null   float64
dtypes: float64(4), int64(6), object(2)
memory usage:

In [26]:
features_sa2_df.isna().sum()

SA2_CODE_2021                      0
SA2_NAME_2021                    106
IRSAD_score                      119
IRSAD_decile                     119
Median_age_persons                 0
Median_mortgage_repay_monthly      0
Median_tot_prsnl_inc_weekly        0
Median_rent_weekly                 0
Median_tot_fam_inc_weekly          0
Average_num_psns_per_bedroom       0
Median_tot_hhd_inc_weekly          0
Average_household_size             0
dtype: int64

In [29]:
# Rename ABS's raw column names 
loaded_features_df = features_sa2_df.rename(columns={
    'Median_tot_prsnl_inc_weekly': 'median_personal_income_weekly',
    'Median_tot_fam_inc_weekly': 'median_family_income_weekly',
    'Median_tot_hhd_inc_weekly': 'median_household_income_weekly',
})

loaded_features_df = loaded_features_df[['SA2_CODE_2021', 'SA2_NAME_2021', 'IRSAD_score', 'IRSAD_decile',
    'median_personal_income_weekly', 'median_family_income_weekly', 'median_household_income_weekly']]

In [30]:
loaded_features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2472 entries, 0 to 2471
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   SA2_CODE_2021                   2472 non-null   object 
 1   SA2_NAME_2021                   2366 non-null   object 
 2   IRSAD_score                     2353 non-null   float64
 3   IRSAD_decile                    2353 non-null   float64
 4   median_personal_income_weekly   2472 non-null   int64  
 5   median_family_income_weekly     2472 non-null   int64  
 6   median_household_income_weekly  2472 non-null   int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 135.3+ KB


In [31]:
# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
loaded_features_df.to_parquet(f"{abs_output_dir}/loaded_features_df.parquet", index=False)